In [ ]:
!pip install tensorflow pandas numpy biopython
!brew install open-babel

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import subprocess
import os
import random
from Bio.PDB import PDBParser
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def prepare_receptor(pdb_file):
    """Prepare the alpha-synuclein fibril for docking"""
    output_pdbqt = pdb_file.replace('.pdb', '.pdbqt')
    subprocess.run(['obabel', pdb_file, '-O', output_pdbqt])
    return output_pdbqt


def prepare_peptide(peptide_sequence):
    """Convert peptide sequence to 3D structure and prepare for docking."""
    
    # Create molecule from peptide sequence
    mol = Chem.MolFromSequence(peptide_sequence)
    if mol is None:
        raise ValueError("Invalid peptide sequence.")
    
    # Add hydrogens and generate 3D coordinates
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    
    # Save the molecule to a PDB file
    pdb_file = f"{peptide_sequence}.pdb"
    with open(pdb_file, 'w') as f:
        f.write(Chem.MolToPDBBlock(mol))
    
    # Convert PDB to PDBQT using Open Babel
    pdbqt_file = f"{peptide_sequence}.pdbqt"
    try:
        pybel.readfile('pdb', pdb_file).write(format='pdbqt', filename=pdbqt_file)
    except Exception as e:
        raise RuntimeError(f"Error converting PDB to PDBQT: {e}")
    return pdbqt_file
# Step 3: Perform molecular docking
def dock_peptide(receptor_file, peptide_file, out_file):
    """Perform molecular docking using AutoDock Vina"""
    subprocess.run(['vina', '--receptor', receptor_file, '--ligand', peptide_file, '--out', out_file])
    
    # Parse the output to get the binding score
    with open(out_file, 'r') as f:
        for line in f:
            if line.startswith('REMARK VINA RESULT:'):
                return float(line.split()[3])
    return None

# Step 4: Extract features from peptides
def extract_features(peptide):
    """Extract relevant features from a peptide sequence"""
    analysis = ProteinAnalysis(peptide)
    mol = Chem.MolFromSequence(peptide)
    return {
        'length': len(peptide),
        'weight': analysis.molecular_weight(),
        'aromaticity': analysis.aromaticity(),
        'instability_index': analysis.instability_index(),
        'isoelectric_point': analysis.isoelectric_point(),
        'hydrophobicity': sum(analysis.protein_scale(window=1, param_dict='kd'))/len(peptide),
        'charge': sum(1 if aa in 'RK' else -1 if aa in 'DE' else 0 for aa in peptide),
        'h_bond_donors': Descriptors.NumHDonors(mol),
        'h_bond_acceptors': Descriptors.NumHAcceptors(mol),
        'rotatable_bonds': Descriptors.NumRotatableBonds(mol),
        'polar_surface_area': Descriptors.TPSA(mol)
    }

# Step 5: Train a machine learning model
def train_model(peptides_df):
    """Train a machine learning model to predict binding scores"""
    X = peptides_df.drop(['sequence', 'binding_score'], axis=1)
    y = peptides_df['binding_score']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    print(f"Model R^2 score on test data: {model.score(X_test_scaled, y_test)}")
    
    return model, scaler

# Step 6: Generate new peptides
def generate_peptide(length=10):
    """Generate a random peptide sequence"""
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    return ''.join(random.choice(amino_acids) for _ in range(length))

def optimize_peptide(model, scaler, base_peptide):
    """Optimize a peptide by making small changes and predicting its binding score"""
    best_peptide = base_peptide
    best_score = predict_binding(model, scaler, base_peptide)
    
    for _ in range(100):  # Try 100 modifications
        new_peptide = list(best_peptide)
        pos = random.randint(0, len(new_peptide) - 1)
        new_peptide[pos] = random.choice('ACDEFGHIKLMNPQRSTVWY')
        new_peptide = ''.join(new_peptide)
        
        new_score = predict_binding(model, scaler, new_peptide)
        if new_score < best_score:  # Lower score is better in docking
            best_peptide = new_peptide
            best_score = new_score
    
    return best_peptide, best_score

def predict_binding(model, scaler, peptide):
    """Predict the binding score for a given peptide"""
    features = pd.DataFrame([extract_features(peptide)])
    features_scaled = scaler.transform(features)
    return model.predict(features_scaled)[0]

def main(a_syn_pdb_file, peptides_csv):
    # Prepare the receptor
    receptor_file = prepare_receptor(a_syn_pdb_file)
    
    # Load and process the peptide data
    peptides_df = pd.read_csv(peptides_csv)
    
    # Perform docking for each peptide
    binding_scores = []
    for peptide in peptides_df['sequence']:
        peptide_file = prepare_peptide(peptide)
        score = dock_peptide(receptor_file, peptide_file, f"{peptide}_docked.pdbqt")
        binding_scores.append(score)
    
    peptides_df['binding_score'] = binding_scores
    
    # Extract features
    features = peptides_df['sequence'].apply(extract_features).apply(pd.Series)
    peptides_df = pd.concat([peptides_df, features], axis=1)
    
    # Train the model
    model, scaler = train_model(peptides_df)
    
    # Generate and optimize new peptides
    new_peptides = []
    for _ in range(10):  # Generate 10 new peptides
        base_peptide = generate_peptide()
        optimized_peptide, predicted_score = optimize_peptide(model, scaler, base_peptide)
        new_peptides.append((optimized_peptide, predicted_score))
    
    # Sort new peptides by predicted binding score
    new_peptides.sort(key=lambda x: x[1])
    
    print("Top 5 predicted stabilizing peptides:")
    for peptide, score in new_peptides[:5]:
        print(f"Sequence: {peptide}, Predicted binding score: {score}")

if __name__ == "__main__":
    a_syn_pdb_file = "data/alpha_synuclein.pdb"
    peptides_csv = "data/peptides.csv"
    main(a_syn_pdb_file, peptides_csv)